In [ ]:
from pathlib import Path
import os
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "rehab" / "data.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Abre Jupyter desde la raíz del repositorio REHAB.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
from rehab.data import DATA_DIR

import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
ruta = DATA_DIR

for npy in ruta.iterdir():
    try:
        sample = np.load(npy, allow_pickle=False)
        print(npy.name)
        print(sample.shape)
    except Exception as e:
        print(f"Error loading {npy.name}: {e}")


014_1.npy esta danado :o

In [ ]:
data = np.load(DATA_DIR / "000_1.npy", allow_pickle=False)

actividad = 0
repeticion = 0
canal = 0  # pitch1

x = data[repeticion, :, canal]

plt.figure(figsize=(10,4))
plt.plot(x)
plt.xlabel("Tiempo (muestra)")
plt.ylabel("Valor")
plt.title("Actividad 000 - Repetición 1 - Canal pitch1")
plt.show()

In [ ]:
ruta = DATA_DIR

canales = [
    "pitch1",
    "yaw1",
    "roll1",
    "pitch2",
    "yaw2",
    "roll2"
]

# Buscar todos los archivos _1.npy excepto el dañado
archivos = sorted(
    archivo for archivo in ruta.glob("*_1.npy")
    if archivo.name != "014_1.npy"
)

print(f"Archivos válidos encontrados: {len(archivos)}")

for archivo in archivos:
    data = np.load(archivo)
    print(f"{archivo.name}: {data.shape}")

In [ ]:
conteos = []
actividades_validas = []

for archivo in archivos:
    data = np.load(archivo)

    actividad = int(archivo.stem.split("_")[0])

    actividades_validas.append(actividad)
    conteos.append(data.shape[0])

tabla_conteos = pd.DataFrame({
    "Actividad": [f"{x:03d}" for x in actividades_validas],
    "Repeticiones": conteos
})

display(tabla_conteos)

In [ ]:
plt.figure(figsize=(12, 5))

plt.bar(
    [f"{x:03d}" for x in actividades_validas],
    conteos
)

plt.xlabel("Actividad")
plt.ylabel("Número de repeticiones")
plt.title("Número de repeticiones por actividad")

plt.show()

In [ ]:
canal = 0

resultados = []

for archivo in archivos:

    actividad = int(archivo.stem.split("_")[0])
    data = np.load(archivo)

    # Concatenar todas las repeticiones y puntos temporales
    valores = data[:, :, canal].flatten()

    resultados.append({
        "Actividad": f"{actividad:03d}",
        "Media": np.mean(valores),
        "Mediana": np.median(valores)
    })

tabla_estadisticas = pd.DataFrame(resultados)

display(tabla_estadisticas)

In [ ]:
def comparar_actividades(actividad_1, actividad_2, canal=0, bins=50):

    archivo_1 = ruta / f"{actividad_1:03d}_1.npy"
    archivo_2 = ruta / f"{actividad_2:03d}_1.npy"

    # Verificar que los archivos existan
    if not archivo_1.exists():
        print(f"No existe el archivo de la actividad {actividad_1:03d}")
        return

    if not archivo_2.exists():
        print(f"No existe el archivo de la actividad {actividad_2:03d}")
        return

    data_1 = np.load(archivo_1)
    data_2 = np.load(archivo_2)

    valores_1 = data_1[:, :, canal].flatten()
    valores_2 = data_2[:, :, canal].flatten()

    media_1 = np.mean(valores_1)
    media_2 = np.mean(valores_2)

    plt.figure(figsize=(10, 5))

    plt.hist(
        valores_1,
        bins=bins,
        alpha=0.5,
        label=f"Actividad {actividad_1:03d}"
    )

    plt.hist(
        valores_2,
        bins=bins,
        alpha=0.5,
        label=f"Actividad {actividad_2:03d}"
    )

    plt.axvline(
        media_1,
        linestyle="--",
        linewidth=2,
        label=f"Media {actividad_1:03d} = {media_1:.2f}"
    )

    plt.axvline(
        media_2,
        linestyle="--",
        linewidth=2,
        label=f"Media {actividad_2:03d} = {media_2:.2f}"
    )

    plt.xlabel(canales[canal])
    plt.ylabel("Frecuencia")

    plt.title(
        f"Actividades {actividad_1:03d} vs {actividad_2:03d} "
        f"- Canal {canales[canal]}"
    )

    plt.legend()
    plt.grid(alpha=0.2)
    plt.show()

In [ ]:
comparar_actividades(0, 1, canal=0, bins=50)

In [ ]:
for canal in range(6):
    comparar_actividades(0, 1, canal=canal, bins=50)